## etcd backup & restore — the CKA's flag-bearer skill

Lose `etcd`, lose the cluster. The CKA tests `etcd snapshot save / restore` reliably — learn it cold.

### Backup — `etcdctl snapshot save`

`etcdctl` needs four auth pieces (certs under `/etc/kubernetes/pki/etcd/`):

```bash
ETCDCTL_API=3 etcdctl snapshot save /backup/snap.db \
  --endpoints=https://127.0.0.1:2379 \
  --cacert=/etc/kubernetes/pki/etcd/ca.crt \
  --cert=/etc/kubernetes/pki/etcd/server.crt \
  --key=/etc/kubernetes/pki/etcd/server.key
```

A snapshot is one file — move it off the node. Verify with `etcdctl --write-out=table snapshot status /backup/snap.db` (revision, key count, hash).

### Restore — `etcdctl snapshot restore`

The standard scenario: "cluster broken, here's a snapshot, restore it."

1. **Stop the control plane** — move the static-pod manifests out of `/etc/kubernetes/manifests/`; the kubelet stops the static pods. Wait for etcd + API server down.
2. **Restore to a *new* data dir** (don't overwrite `/var/lib/etcd`):
   ```bash
   ETCDCTL_API=3 etcdctl snapshot restore /backup/snap.db --data-dir=/var/lib/etcd-from-backup
   ```
3. **Repoint etcd** — in `etcd.yaml`, change the `etcd-data` `hostPath` from `/var/lib/etcd` to `/var/lib/etcd-from-backup`.
4. **Restore the manifests** — move them back; the kubelet boots etcd from the snapshot.
5. **Verify** — `kubectl get nodes`, `kubectl get pods -A`.

Memorise: **`ETCDCTL_API=3`** (default in 3.5+); **three certs** (ca/cert/key); a snapshot is **point-in-time** (post-snapshot changes are lost); external etcd is the same recipe with a different `--endpoints`. On our map this is the operational lifeline for the **etcd** component — the difference between a bad hour and a dead cluster.